[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Imsharad/ghl-support-slm/blob/v2/notebooks/train_colab.ipynb)

# GoHighLevel support SLM, v2: QLoRA on Qwen2.5-1.5B-Instruct, free T4

v1 of this project fine-tuned Qwen2.5-1.5B-Instruct on the Bitext customer-support corpus and lost a blind evaluation: the cleaner had thrown away every row that admits a missing fact, so the tuned model learned the corpus voice and stopped saying "I do not have that". v2 changes the data and nothing else: placeholders are substituted with neutral phrases instead of rejecting the row, and 340 synthetic admission rows are appended to train. Base model, QLoRA recipe, prompt, decoding, rubric and judge protocol are frozen.

This notebook is the v2 training run. Top to bottom on a free T4 it takes about 1.5 hours and produces `train/runs/v2-t4/`: the adapter checkpoints, `loss.csv`, `curves.png`, per-checkpoint dev answers and a gate report. Scoring stays on the sealed Mac path (Q8 GGUF through Ollama), the same pipeline that scored v1.

Repo: [github.com/Imsharad/ghl-support-slm](https://github.com/Imsharad/ghl-support-slm), branch `v2`. Plan and decisions: `docs/plans/V2_PLAN.md`, `docs/v2/PLAN_DECISIONS.md`.

## 1. Runtime

In [ ]:
import os, subprocess, time, json, shutil
os.environ.update(HF_HUB_DISABLE_PROGRESS_BARS="1", HF_HUB_DISABLE_IMPLICIT_TOKEN="1", TOKENIZERS_PARALLELISM="false", PYTHONUNBUFFERED="1")

def run(cmd, tail=None, quiet=("it/s]", "s/it]", "Warning", "warn(", "Could not load this library", "Failed to load /usr")):
    """Run a shell command, print its output without progress bars, stop the notebook on failure."""
    t = time.time()
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    lines = [l for l in (p.stdout + p.stderr).splitlines() if l.strip() and not any(q in l for q in quiet)]
    print("\n".join(lines[-tail:] if tail else lines))
    print(f"[{cmd.split()[0]} rc={p.returncode} {time.time()-t:.0f}s]")
    if p.returncode: raise SystemExit(p.returncode)

run("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader && python -V")

## 2. Pinned versions

The same versions as `configs/models/versions.json` and `uv.lock`. Colab's preinstalled `torchao` (0.10) breaks adapter reload under transformers 5.16, so it is upgraded too.

In [ ]:
run("pip install -q 'transformers==5.16.1' 'peft==0.20.0' 'trl==1.12.0' 'bitsandbytes==0.50.2' 'accelerate>=0.34' "
    "'datasets==5.0.1' 'sentence-transformers>=3.0' 'scikit-learn>=1.5' 'pyyaml>=6.0' 'matplotlib>=3.9' 'torchao>=0.16'")
run("python -c \"import torch, transformers, peft, bitsandbytes, trl, torchao; "
    "print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| transformers', transformers.__version__, "
    "'| peft', peft.__version__, '| bitsandbytes', bitsandbytes.__version__, '| trl', trl.__version__, '| torchao', torchao.__version__)\"")

## 3. Code at a pinned commit

The repo is public. `COMMIT` pins the exact tree this run trained from; it is recorded again in `config.json` by the trainer.

In [ ]:
REPO, BRANCH, COMMIT = "https://github.com/Imsharad/ghl-support-slm.git", "v2", "origin/v2"
WORKDIR = "/content/ghl-support-slm"
if not os.path.isdir(WORKDIR):
    run(f"git clone -q --branch {BRANCH} {REPO} {WORKDIR}")
os.chdir(WORKDIR); os.environ["PYTHONPATH"] = WORKDIR  # eval/ and tools/ import as packages from the repo root
run(f"git fetch -q origin {BRANCH} && git checkout -q --detach {COMMIT} && git log --oneline -1")

## 4. Pinned base model

`train/render.py` and `tools/training/check_run.py` load the base tokenizer and weights from the local Hugging Face cache only (`local_files_only=True`), so the pinned snapshot is fetched once here.

In [ ]:
from huggingface_hub import snapshot_download
pin = json.load(open("configs/models/versions.json"))["base_model"]
path = snapshot_download(pin["repo_id"], revision=pin["revision"])
print(pin["repo_id"], "@", pin["revision"][:8], "->", path)

## 5. Data: fetch, clean in substitution mode, append the admission rows, audit

`data/prepare.py --placeholder-mode substitute` is the v2 cleaner (decision 1 to 4 in `PLAN_DECISIONS.md`). `--admissions` appends the 340 reviewed synthetic rows, flagged `SYNTH_ADMISSION`, to train only, outside the 8,000-row cap. The strict audit recomputes every split hash and the group and text intersections; it exits non-zero on any mismatch.

In [ ]:
run("python data/fetch.py", tail=3)
run("python data/prepare.py --placeholder-mode substitute --out data/v2 --admissions data/v2/admissions.jsonl", tail=8)
run("python data/prepare.py --audit-only --out data/v2 --strict")
run("wc -l data/v2/admissions.jsonl data/processed/v2/train.jsonl data/processed/v2/val.jsonl data/processed/v2/test.jsonl")
run("sha256sum data/v2/admissions.jsonl data/processed/v2/train.jsonl data/processed/v2/val.jsonl data/processed/v2/test.jsonl | cut -c1-16,64-")

## 6. Configuration, frozen

`configs/training/train-t4.yaml` is the config the served v1 checkpoint was trained on. Nothing in it changes for v2; the only variable between the two runs is the data.

In [ ]:
import yaml
cfg = yaml.safe_load(open("configs/training/train-t4.yaml"))
show = {"seed": cfg["seed"], "cap_train": cfg["data"]["cap_train"], "max_length": cfg["data"]["max_length"],
        "load_in_4bit": cfg["model"]["load_in_4bit"], "quant_type": cfg["model"]["quant_type"], "compute_dtype": cfg["model"]["compute_dtype"],
        "lora": {k: cfg["lora"][k] for k in ("r", "alpha", "dropout")}, "target_modules": cfg["lora"]["target_modules"],
        "lr": cfg["optim"]["lr"], "schedule": cfg["optim"]["schedule"], "warmup_ratio": cfg["optim"]["warmup_ratio"],
        "epochs": cfg["train"]["epochs"], "micro_batch_size": cfg["train"]["micro_batch_size"], "grad_accum": cfg["train"]["grad_accum"],
        "checkpoint_fractions": cfg["train"]["checkpoint_fractions"], "save_every": cfg["train"]["save_every"]}
print(json.dumps(show, indent=1))

## 7. Smoke run with a resume proof

Twenty steps on 64 rows, then a resume from step 10 that must reproduce steps 11 to 20 within tolerance. `tools/training/check_run.py` reloads the adapter and generates once.

`run()` raises on a non-zero exit, so an interactive Run-all stops here and nothing below executes. A batch execution under `--ExecutePreprocessor.allow_errors=True` does not stop: read this cell's output before trusting a batch-executed copy. `notebooks/v2_colab_run.ipynb` is one such copy and this cell failed in it; see the note in that notebook and README section 7.

In [ ]:
shutil.rmtree("train/runs/v2-t4", ignore_errors=True)
run("python train/train.py --config configs/training/train-t4.yaml --smoke --data-dir data/processed/v2 --run-name v2-t4 --no-push", tail=4)
s = json.load(open("train/runs/v2-t4/smoke.json")); print({k: s[k] for k in ("device", "steps", "resume_from", "max_abs_diff", "ok")})
run("python tools/training/check_run.py train/runs/v2-t4 --max-memory-gb 12")
shutil.rmtree("train/runs/v2-t4")

## 8. Train

One epoch over the capped v2 train split. The trainer logs every 25 steps and saves checkpoints at steps 250, 300, 400 and 500 plus the end of the epoch. If the session drops, re-run this cell with `--resume`.

In [ ]:
run("python train/train.py --config configs/training/train-t4.yaml --data-dir data/processed/v2 --run-name v2-t4 --no-push")

## 9. Loss curve and the completeness gate

In [ ]:
run("python train/plot_curves.py train/runs/v2-t4")
from IPython.display import Image, display
display(Image("train/runs/v2-t4/curves.png", width=720))
run("python tools/training/check_run.py train/runs/v2-t4 --max-memory-gb 12")
c = json.load(open("train/runs/v2-t4/config.json"))
print({k: c.get(k) for k in ("run_name", "device", "git_sha", "final_step", "wall_s", "peak_memory_gb")})
run("tail -n 6 train/runs/v2-t4/loss.csv")

## 10. Dev answers for every checkpoint

Checkpoint selection is made on the 54-item dev set only, never on a sealed set (`docs/SELECTION.md`). Each checkpoint is merged into fp16, answers dev through the transformers backend, and the answers are kept with the run.

In [ ]:
from pathlib import Path
run_dir = Path("train/runs/v2-t4")
ckpts = sorted((int(p.name.split("-")[1]), p) for p in run_dir.glob("checkpoint-*") if (p / "adapter_config.json").is_file())
print("checkpoints:", [s for s, _ in ckpts])
for step, folder in ckpts:
    out = run_dir / f"dev-checkpoint-{step}-raw.jsonl"
    shutil.rmtree("artifacts/merged", ignore_errors=True)
    run(f"python tools/artifacts/merge.py --adapter {folder} --output artifacts/merged", tail=1)
    run(f"python eval/run.py --model tuned --backend transformers --split dev --device cuda --output {out} --check-complete", tail=2)
shutil.rmtree("artifacts/merged", ignore_errors=True)

## 11. Package the run

Everything under `train/runs/v2-t4/` in one zip with its hash. The Mac side unpacks it, selects the checkpoint on dev, merges, quantizes to Q8, tags it in Ollama and scores it on the fresh sealed set.

In [ ]:
archive = shutil.make_archive("/content/v2-t4", "zip", run_dir)
run(f"ls -la {archive} && sha256sum {archive}")
try:
    from google.colab import files; files.download(archive)
except Exception:
    print("not an interactive Colab kernel; download the zip with `colab download /content/v2-t4.zip <local>`")

## What this run is, and is not

- It is the v2 headline training substrate if it finished clean here (gate above) and its artifacts reached the scoring machine in time; the rule was fixed before the run started.
- It is not the evaluation. The blind pass rate on the fresh sealed set comes from the sealed Ollama path on the Mac, judged by the frozen protocol, and is reported in `docs/RESULTS.md` beside v1's.